# Learning a random reversible circuit: train/eval over time

Reproducibly samples a 256-wire random reversible brickwork circuit with
uniform output tap depths (see `README.md`), then trains a pre-norm residual
MLP online: **every step draws a fresh batch of inputs** (sampling with
replacement from $\{0,1\}^{256}$ — collision odds are nil), so there is no
fixed training set and no memorization; train loss is itself an unbiased
generalization estimate, and the held-out eval set just measures it with lower
variance on a fixed sample.

Defaults: 50k steps, eval on 1000 fixed samples every 500 steps
(~15 min on CPU at the default model size). All randomness is seeded:
the circuit, model init, eval set, and the training-batch stream
(batch at step $t$ is derived by folding $t$ into `DATA_SEED`, so runs are
reproducible and restartable).

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import optax
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from random_circuit import sample_circuit, make_jax_evaluator
from mlp import MLPConfig, init_params, forward, per_output_bce

# --- seeds ---
CIRCUIT_SEED = 0
MODEL_SEED = 0
DATA_SEED = 0    # stream of fresh training batches
EVAL_SEED = 1    # fixed held-out eval inputs

# --- circuit ---
N_WIRES = 256
CIRC_DEPTH = 32

# --- model / training ---
WIDTH, MLP_DEPTH = 256, 4   # ~2.2M params, ~17 ms/step on CPU
BATCH = 256
STEPS = 50_000
EVAL_EVERY = 500
EVAL_N = 1000
PEAK_LR = 1e-3

## Circuit

In [ ]:
rng = np.random.default_rng(CIRCUIT_SEED)
circuit = sample_circuit(rng, n_wires=N_WIRES, depth=CIRC_DEPTH)
circ_eval = make_jax_evaluator(circuit)
out_depths = np.asarray(circuit.out_depths)

n_gates = sum(
    int((circuit.tables[t, k] != np.arange(8)).any())
    for t in range(CIRC_DEPTH)
    for k in range(circuit.n_gate_slots)
)
print(f"{N_WIRES} wires, {CIRC_DEPTH} layers, {n_gates} gates")
plt.figure(figsize=(5, 2))
plt.hist(out_depths, bins=np.arange(0.5, CIRC_DEPTH + 1.5), color="tab:blue")
plt.xlabel("output tap depth"); plt.ylabel("# outputs"); plt.tight_layout()

## Model, data stream, train/eval steps

In [ ]:
cfg = MLPConfig(
    n_inputs=N_WIRES, n_outputs=N_WIRES, width=WIDTH, depth=MLP_DEPTH
)
params = init_params(jax.random.key(MODEL_SEED), cfg)
n_params = sum(p.size for p in jax.tree_util.tree_leaves(params))
print(f"model: width={WIDTH} depth={MLP_DEPTH} ({n_params/1e6:.2f}M params)")

schedule = optax.warmup_cosine_decay_schedule(
    init_value=0.0, peak_value=PEAK_LR, warmup_steps=500,
    decay_steps=STEPS, end_value=0.1 * PEAK_LR,
)
opt = optax.adam(schedule)
opt_state = opt.init(params)

data_key = jax.random.key(DATA_SEED)


@jax.jit
def train_step(params, opt_state, step):
    key = jax.random.fold_in(data_key, step)
    x = jax.random.bernoulli(key, shape=(BATCH, N_WIRES)).astype(jnp.uint8)
    y = circ_eval(x).astype(jnp.float32)

    def loss_fn(p):
        return jnp.mean(per_output_bce(forward(p, x), y))

    loss, grads = jax.value_and_grad(loss_fn)(params)
    updates, opt_state = opt.update(grads, opt_state)
    return optax.apply_updates(params, updates), opt_state, loss


eval_x = jax.random.bernoulli(
    jax.random.key(EVAL_SEED), shape=(EVAL_N, N_WIRES)
).astype(jnp.uint8)
eval_y = circ_eval(eval_x).astype(jnp.float32)


@jax.jit
def evaluate(params):
    logits = forward(params, eval_x)
    per_out_loss = per_output_bce(logits, eval_y)
    per_out_acc = jnp.mean((logits > 0) == (eval_y > 0.5), axis=0)
    return per_out_loss, per_out_acc

## Train

In [ ]:
train_loss = np.zeros(STEPS)
eval_steps, eval_loss, eval_acc = [], [], []
per_out_loss_hist, per_out_acc_hist = [], []


def run_eval(step):
    pol, poa = jax.device_get(evaluate(params))
    eval_steps.append(step)
    eval_loss.append(pol.mean())
    eval_acc.append(poa.mean())
    per_out_loss_hist.append(pol)
    per_out_acc_hist.append(poa)


pbar = tqdm(range(STEPS))
for step in pbar:
    if step % EVAL_EVERY == 0:
        run_eval(step)
    params, opt_state, loss = train_step(params, opt_state, step)
    train_loss[step] = float(loss)
    if step % 100 == 0:
        pbar.set_postfix(train=f"{train_loss[step]:.4f}", eval_acc=f"{eval_acc[-1]:.3f}")
run_eval(STEPS)

per_out_loss_hist = np.array(per_out_loss_hist)  # (n_evals, N_WIRES)
per_out_acc_hist = np.array(per_out_acc_hist)
eval_steps = np.array(eval_steps)
print(f"final: eval loss {eval_loss[-1]:.4f}, eval acc {eval_acc[-1]:.4f}")

## Train / eval loss and accuracy over time

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

window = 100
smoothed = np.convolve(train_loss, np.ones(window) / window, mode="valid")
axes[0].plot(np.arange(len(smoothed)) + window // 2, smoothed,
             label=f"train (rolling {window})", lw=1)
axes[0].plot(eval_steps, eval_loss, label="eval (1000 fixed samples)", lw=1.5)
axes[0].axhline(np.log(2), color="gray", ls=":", lw=1, label="chance (ln 2)")
axes[0].set(xlabel="step", ylabel="BCE / output bit", yscale="log")
axes[0].legend(); axes[0].set_title("loss")

axes[1].plot(eval_steps, eval_acc, lw=1.5, color="tab:green")
axes[1].axhline(0.5, color="gray", ls=":", lw=1)
axes[1].set(xlabel="step", ylabel="mean per-bit accuracy", ylim=(0.45, 1.02))
axes[1].set_title("eval accuracy")
plt.tight_layout()

## Depth-resolved learning curves

Each output's tap depth is a hardness dial: shallow outputs depend on few
inputs and are learned quickly; deep outputs approach a random permutation's
bits and stay at chance at this model scale.

In [ ]:
buckets = [(1, 2), (3, 4), (5, 6), (7, 8), (9, 12), (13, 32)]
colors = plt.cm.viridis(np.linspace(0, 0.9, len(buckets)))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for (lo, hi), c in zip(buckets, colors):
    sel = (out_depths >= lo) & (out_depths <= hi)
    axes[0].plot(eval_steps, per_out_acc_hist[:, sel].mean(axis=1),
                 color=c, lw=1.5, label=f"depth {lo}-{hi} (n={sel.sum()})")
axes[0].axhline(0.5, color="gray", ls=":", lw=1)
axes[0].set(xlabel="step", ylabel="eval accuracy")
axes[0].legend(fontsize=8); axes[0].set_title("accuracy by output tap depth")

order = np.argsort(out_depths)
axes[1].scatter(out_depths[order], per_out_acc_hist[-1][order], s=12, alpha=0.6)
axes[1].axhline(0.5, color="gray", ls=":", lw=1)
axes[1].set(xlabel="output tap depth", ylabel="final eval accuracy")
axes[1].set_title(f"final accuracy vs depth (step {eval_steps[-1]})")
plt.tight_layout()

## Save history

In [ ]:
run_name = f"w{WIDTH}_d{MLP_DEPTH}_b{BATCH}_s{STEPS}_seed{MODEL_SEED}"
np.savez_compressed(
    f"history_{run_name}.npz",
    train_loss=train_loss,
    eval_steps=eval_steps,
    per_out_loss=per_out_loss_hist,
    per_out_acc=per_out_acc_hist,
    out_depths=out_depths,
    config=np.array(
        [CIRCUIT_SEED, MODEL_SEED, DATA_SEED, EVAL_SEED, N_WIRES, CIRC_DEPTH,
         WIDTH, MLP_DEPTH, BATCH, STEPS, EVAL_EVERY, EVAL_N]
    ),
)
print(f"saved history_{run_name}.npz")